This notebook only builds the persistent ChromaDB index. Later Python scripts will load the same embedding model for retrieval and use a cross-encoder, LangGraph, Ollama, and FastAPI.

**Design:** ~120 BGE-token chunks, no overlap; for each chunk, embed its previous + current + next chunk (within the same book). Store the original center chunk as the document and book/page/chunk index as metadata. The contextual vector represents the entire three-chunk window, not only the center chunk.

Run cells from top to bottom. Re-running the indexing cell uses `upsert`, so it updates existing IDs rather than duplicating them. If you change the chunking or model, change the collection name or delete the old collection first.


In [1]:
import sys
print(sys.executable)

d:\envs\env1\env1\Scripts\python.exe


1- config and import

env: `D:\envs\env1\env1` 

In [2]:
from pathlib import Path
import re
import sys

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

BOOKS_DIR = Path(r"D:\Documents\books\hp")
PROJECT_DIR = Path(r"D:\projects\langgraph_rag_harrypotter")
DB_DIR = PROJECT_DIR / "chroma_db"

MODEL_NAME = "BAAI/bge-base-en-v1.5"
COLLECTION_NAME = "hp_bge_contextual_120_v1"
CHUNK_SIZE = 120
BATCH_SIZE = 16

print("Python:", sys.executable)
print("Books directory exists:", BOOKS_DIR.is_dir())
print("Database directory:", DB_DIR)

d:\envs\env1\env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: d:\envs\env1\env1\Scripts\python.exe
Books directory exists: True
Database directory: D:\projects\langgraph_rag_harrypotter\chroma_db


2-  verify the seven PDFs

In [3]:
pdf_files = sorted(
    BOOKS_DIR.glob("Book *.pdf"),
    key=lambda p: int(re.match(r"Book (\d+)", p.name).group(1))
)

assert len(pdf_files) == 7, f"Expected 7 PDFs, found {len(pdf_files)}"
for pdf in pdf_files:
    print(pdf.name)

Book 1 - Harry Potter and the Sorcerers Stone.pdf
Book 2 - Harry Potter and the Chamber of Secrets.pdf
Book 3 - Harry Potter and the Prisoner of Azkaban.pdf
Book 4 - Harry Potter and the Goblet of Fire.pdf
Book 5 - Harry Potter and the Order of the Phoenix.pdf
Book 6 - Harry Potter and the Half-Blood Prince.pdf
Book 7 - Harry Potter and the Deathly Hallows.pdf


3- extract text with page numbers

we keep page metadata for citations later. `pypdf` works for text-based PDFs not scanned ones needing OCR

In [4]:
def read_book(pdf_path):
    """Extract readable text from each PDF page."""
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"page": page_num, "text": text})
    return pages

# Inspect only the first book before processing all seven.
sample_pages = read_book(pdf_files[0])
print("Text pages in first book:", len(sample_pages))
print("Preview:", sample_pages[0]["text"][:900])

Text pages in first book: 206
Preview: Harry Potter and the Sorcerer’s Stone By J.K. Rowling CHAPTER ONE The Boy Who Lived Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much. They were the last people you’d expect to be involved in anything strange or mysterious, because they just didn’t hold with such nonsense. Mr. Dursley was the director of a firm called Grunnings, which made drills. He was a big, beefy man with hardly any neck, although he did have a very large mustache. Mrs. Dursley was thin and blonde and had nearly twice the usual amount of neck, which came in very useful as she spent so much of her time craning over garden fences, spying on the neighbors. The Dursleys had a small son called Dudley and in their opinion there was no finer boy anywhere. The Dursleys had everything they wanted, but they also had a secret, and their greatest fear was t


4- Load BGE 

The encoder uses self-attention within the combined context text. We do not train an extra attention layer. All embeddings and queries must use the same model.

In [5]:
model = SentenceTransformer(MODEL_NAME, device="cpu")
tokenizer = model.tokenizer
MAX_TOKENS = model.max_seq_length
print("Embedding model:", MODEL_NAME)
print("Max input tokens:", MAX_TOKENS)
print("Embedding dimensions:", model.get_sentence_embedding_dimension())

d:\envs\env1\env1\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASus\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\envs\env1\env1\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlink

Embedding model: BAAI/bge-base-en-v1.5
Max input tokens: 512
Embedding dimensions: 768


C:\Users\ASus\AppData\Local\Temp\ipykernel_28956\426221173.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


5- create chunks and contextual windows

Each chunk has at most 120 BGE tokens and is tied to its starting PDF page. We split within pages for straightforward page metadata. this may produce short end-of-page chunks and split sentences. Neighboring chunks might cross pages but they dont cross books. We check the exact combined token count including special tokens before encoding.

6- extract and validate all seven books
ddd
This cell does not write to ChromaDB. It checks that all books have readable text and that every three-chunk context fits the model input limit. If any window is too long, stop and adjust the chunking strategy rather than silently truncating.